# Intro a REST

En este ejercicio se implementó una API REST utilizando Django y el modelo
ProductModel almacenado en la base de datos.

Se implementaron y probaron los métodos HTTP:

- GET - Consultar o listar productos.
- POST - Crear productos.
- PUT - Actualizar productos.
- DELETE - Eliminar productos.

Los métodos realizan cambios reales sobre los registros almacenados en la
base de datos.


## API REST - ejercicios/api.py

In [ ]:
import json

from django.http import JsonResponse
from django.views.decorators.csrf import csrf_exempt

from .models import ProductModel


@csrf_exempt
def products_api(request, product_id=None):

    # GET - Consultar o listar productos
    if request.method == "GET":
        print("Método GET ejecutado")

        if product_id is not None:
            try:
                product = ProductModel.objects.get(pk=product_id)
            except ProductModel.DoesNotExist:
                return JsonResponse(
                    {"error": "Producto no encontrado"},
                    status=404,
                )

            data = {
                "id": product.id,
                "name": product.name,
                "price": str(product.price),
                "description": product.description,
                "seller": product.seller,
                "color": product.color,
                "product_dimensions": product.product_dimensions,
            }

            return JsonResponse(data)

        products = list(
            ProductModel.objects.values(
                "id",
                "name",
                "price",
                "description",
                "seller",
                "color",
                "product_dimensions",
            )
        )

        return JsonResponse(
            {"products": products}
        )

    # POST - Crear un producto
    if request.method == "POST":
        print("Método POST ejecutado")

        try:
            data = json.loads(request.body)

            product = ProductModel.objects.create(
                name=data["name"],
                price=data["price"],
                description=data["description"],
                seller=data["seller"],
                color=data["color"],
                product_dimensions=data["product_dimensions"],
            )

            return JsonResponse(
                {
                    "message": "Producto creado correctamente",
                    "id": product.id,
                },
                status=201,
            )

        except (json.JSONDecodeError, KeyError):
            return JsonResponse(
                {"error": "Datos inválidos"},
                status=400,
            )

    # PUT - Actualizar un producto
    if request.method == "PUT":
        print("Método PUT ejecutado")

        if product_id is None:
            return JsonResponse(
                {"error": "Debes indicar el ID del producto"},
                status=400,
            )

        try:
            product = ProductModel.objects.get(pk=product_id)
        except ProductModel.DoesNotExist:
            return JsonResponse(
                {"error": "Producto no encontrado"},
                status=404,
            )

        try:
            data = json.loads(request.body)

            product.name = data.get("name", product.name)
            product.price = data.get("price", product.price)
            product.description = data.get(
                "description",
                product.description,
            )
            product.seller = data.get(
                "seller",
                product.seller,
            )
            product.color = data.get(
                "color",
                product.color,
            )
            product.product_dimensions = data.get(
                "product_dimensions",
                product.product_dimensions,
            )

            product.save()

            return JsonResponse(
                {
                    "message": "Producto actualizado correctamente",
                    "id": product.id,
                }
            )

        except json.JSONDecodeError:
            return JsonResponse(
                {"error": "JSON inválido"},
                status=400,
            )

    # DELETE - Eliminar un producto
    if request.method == "DELETE":
        print("Método DELETE ejecutado")

        if product_id is None:
            return JsonResponse(
                {"error": "Debes indicar el ID del producto"},
                status=400,
            )

        try:
            product = ProductModel.objects.get(pk=product_id)
        except ProductModel.DoesNotExist:
            return JsonResponse(
                {"error": "Producto no encontrado"},
                status=404,
            )

        product.delete()

        return JsonResponse(
            {"message": "Producto eliminado correctamente"}
        )

    return JsonResponse(
        {"error": "Método HTTP no permitido"},
        status=405,
    )


## Rutas de la API

In [ ]:
from ejercicios.api import products_api

urlpatterns += [
    path(
        "api/products/",
        products_api,
        name="api-products",
    ),

    path(
        "api/products/<int:product_id>/",
        products_api,
        name="api-product-detail",
    ),
]


## Pruebas realizadas

### GET

Se consultó un producto existente mediante:

    GET /api/products/2/

Resultado:

    {
        "id": 2,
        "name": "Producto 1",
        "price": "101.00"
    }

El método GET consultó correctamente un registro almacenado en la base de datos.

### POST

Se creó un nuevo producto mediante:

    POST /api/products/

Se enviaron los datos:

    {
        "name": "Producto REST",
        "price": "599.99",
        "description": "Producto creado mediante POST",
        "seller": "API REST",
        "color": "Negro",
        "product_dimensions": "20 x 10 x 5 cm"
    }

Resultado:

    {
        "message": "Producto creado correctamente",
        "id": 502
    }

Posteriormente se utilizó GET sobre el producto 502 para comprobar que el
registro había sido almacenado realmente en la base de datos.

### PUT

Se actualizó el producto 502 mediante:

    PUT /api/products/502/

Se modificaron los siguientes datos:

    {
        "name": "Producto REST Actualizado",
        "price": "749.99",
        "description": "Producto modificado mediante PUT",
        "color": "Azul"
    }

Resultado:

    {
        "message": "Producto actualizado correctamente",
        "id": 502
    }

Después se realizó un GET y se comprobó que los nuevos valores habían quedado
guardados en la base de datos.

### DELETE

Se eliminó el producto mediante:

    DELETE /api/products/502/

Resultado:

    {
        "message": "Producto eliminado correctamente"
    }

Finalmente se volvió a realizar un GET sobre el producto 502.

El servidor respondió:

    HTTP 404 Not Found

    {
        "error": "Producto no encontrado"
    }

Esto comprobó que el registro había sido eliminado realmente de la base de datos.


## Resultado

La API implementa correctamente los cuatro métodos HTTP solicitados:

- POST: crea registros reales en ProductModel.
- GET: consulta o lista registros almacenados.
- PUT: modifica registros existentes.
- DELETE: elimina registros de la base de datos.

También se utilizaron prints para identificar la ejecución de cada método,
siguiendo la implementación mostrada en el módulo.
